In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import scipy.stats as stats
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import lightgbm as lgb

In [11]:
df = pd.read_parquet('feature.parq')
df.head()

,mid_prc,b66accf4d4,01d830fc33,ac29aa28b0,86eecbe036,9c51f3cf1d,706dbe6d28,cb232e1c9f,301f8d1b44,134e1a6937,...,2d1cf32644,18af9014b6,9b980c18d7,d762c12f50,c7c3c1666c,73762306aa,1c6e322c2d,0d19cb3fb0,486cb65b0c,c49a5f9c65
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-04-16 00:00:05,1588.010010,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.000000,...,5.000000,5.000000,5.000000,5.0,5.000000,5.000000,5.000000,5.0,5.000000,5.0000
2025-04-16 00:00:06,1587.824951,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,0.601331,...,0.023879,-1.795081,5.000000,5.0,-0.614067,-5.000000,5.000000,5.0,-0.774996,-5.0000
2025-04-16 00:00:07,1587.824951,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,0.322740,...,0.766593,5.000000,5.000000,5.0,0.142544,3.931395,5.000000,5.0,-0.024244,-3.6261
2025-04-16 00:00:08,1587.305054,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.000000,...,-5.000000,-5.000000,-4.450434,-5.0,-5.000000,-5.000000,-4.419014,-5.0,-5.000000,-5.0000
2025-04-16 00:00:09,1587.285034,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.000000,...,-5.000000,-5.000000,-4.941638,-5.0,-5.000000,-5.000000,-4.966719,-5.0,-5.000000,-5.0000


In [12]:
df.describe()

,mid_prc,b66accf4d4,01d830fc33,ac29aa28b0,86eecbe036,9c51f3cf1d,706dbe6d28,cb232e1c9f,301f8d1b44,134e1a6937,...,2d1cf32644,18af9014b6,9b980c18d7,d762c12f50,c7c3c1666c,73762306aa,1c6e322c2d,0d19cb3fb0,486cb65b0c,c49a5f9c65
count,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,...,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000,604378.000000
mean,1601.856689,-0.001573,0.000471,0.000961,0.000961,-0.003787,-0.004580,-0.002970,-0.003124,0.000960,...,0.000102,-0.001177,-0.001475,-0.001235,-0.000627,-0.001408,-0.001800,-0.001257,-0.000812,-0.001446
std,33.063606,1.002640,1.000666,0.999296,0.999296,1.005991,1.007335,1.004184,1.002396,1.061816,...,0.997693,1.001713,0.967141,1.001705,1.002303,1.001646,0.966189,1.001703,1.002400,1.001568
min,1536.010010,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,...,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000
25%,1581.724976,-0.673479,-0.634930,-0.610623,-0.610618,-0.661305,-0.605361,-0.596674,-0.595345,-0.295918,...,-0.523161,-1.018428,-0.643154,-1.028844,-0.588139,-1.011383,-0.636527,-1.028464,-0.621631,-1.004722
50%,1591.655029,-0.007083,-0.004322,0.000993,0.001000,-0.003068,-0.006797,0.000932,0.000944,-0.000310,...,0.009299,0.770835,0.010962,0.779418,0.011513,0.718494,0.008375,0.769727,0.011204,0.690965
75%,1612.604980,0.664406,0.633471,0.612619,0.612619,0.654505,0.595105,0.594139,0.592611,0.303442,...,0.532350,0.960790,0.643316,0.957942,0.592339,0.947430,0.634845,0.954898,0.628167,0.936125
max,1777.680054,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 604378 entries, 2025-04-16 00:00:05 to 2025-04-22 23:59:59
Data columns (total 87 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   mid_prc     604378 non-null  float32
 1   b66accf4d4  604378 non-null  float32
 2   01d830fc33  604378 non-null  float32
 3   ac29aa28b0  604378 non-null  float32
 4   86eecbe036  604378 non-null  float32
 5   9c51f3cf1d  604378 non-null  float32
 6   706dbe6d28  604378 non-null  float32
 7   cb232e1c9f  604378 non-null  float32
 8   301f8d1b44  604378 non-null  float32
 9   134e1a6937  604378 non-null  float32
 10  5f3f783a1f  604378 non-null  float32
 11  d182b35ca9  604378 non-null  float32
 12  c910253362  604378 non-null  float32
 13  8572904679  604378 non-null  float32
 14  486b27acac  604378 non-null  float32
 15  e10ab80234  604378 non-null  float32
 16  5e7e5a691e  604378 non-null  float32
 17  ec021e3c39  604378 non-null  float32
 18  c12d090869

In [15]:
# Remove NaN values
df_clean = df.dropna()

# Extract features (all columns except price and target)
feature_cols = [col for col in df_clean.columns if col not in ['mid_prc']]

# Calculate log returns for target variable (you can adjust the horizon)
horizon = [1, 2, 5, 10, 30, 60, 300, 600]  # 5-second forward returns

for h in horizon:
    df_clean[f'target_{h}'] = np.log(df_clean['mid_prc'] / df_clean['mid_prc'].shift(h)).shift(-h)

# Time-series split (no shuffling to preserve temporal order)
split_idx = int(len(df_clean) * 0.6)  # 60% for training
split2_idx = int(len(df_clean) * 0.8)  # 20% for validation, 20% for testing
train_df = df_clean.iloc[:split_idx]
test_df = df_clean.iloc[split_idx:split2_idx]
val_df = df_clean.iloc[split2_idx:]

#check splits
print(f'Training set: {train_df.index.min()} - {train_df.index.max()}')
print(f'Test set: {test_df.index.min()} - {test_df.index.max()}')
print(f'Validation set: {val_df.index.min()} - {val_df.index.max()}')

Training set: 2025-04-16 00:00:05 - 2025-04-20 04:49:09
Test set: 2025-04-20 04:49:10 - 2025-04-21 14:24:50
Validation set: 2025-04-21 14:24:51 - 2025-04-22 23:59:59
